#Silver Layer ETL Pipeline
Digital Lifestyle Benchmark Dataset — Medallion Architecture

This notebook implements the Silver Layer for the Digital Lifestyle Benchmark dataset.

The Bronze dataset is loaded and validated. Data quality checks are performed, categorical values are standardized, business rules are applied, and analytical features are created.

The final output is saved as:

Silver_Layer/Digital_Lifestyle_Benchmark_Silver.csv

This dataset will be used as the source for the Gold Layer.

##1. Define File Paths & Create Silver Layer Folder

In [ ]:
BRONZE_FILE_PATH = "Bronze_Layer/Digital_Lifestyle_Benchmark_Bronze.csv"
SILVER_FOLDER = "Silver_Layer"
SILVER_FILE_NAME = "Digital_Lifestyle_Benchmark_Silver.csv"
SILVER_FILE_PATH = os.path.join( SILVER_FOLDER, SILVER_FILE_NAME )

os.makedirs(SILVER_FOLDER, exist_ok=True)
print(f"Silver Layer folder ready at: ./{SILVER_FOLDER}/")

Silver Layer folder ready at: ./Silver_Layer/


##2. Load Bronze Dataset

In [ ]:
df = pd.read_csv(BRONZE_FILE_PATH)

print("Bronze dataset loaded successfully.")
print(df.shape)

Bronze dataset loaded successfully.
(100000, 24)


##3. Data Quality Validation

In [ ]:
print("===== DATA VALIDATION =====")

print("\nMissing Values:")
print(df.isnull().sum())

print("\nDuplicate IDs:")
print(df["id"].duplicated().sum())

print("\nDataset Shape:")
print(df.shape)

===== DATA VALIDATION =====

Missing Values:
id                          0
age                         0
gender                      0
region                      0
income_level                0
education_level             0
daily_role                  0
device_hours_per_day        0
phone_unlocks               0
notifications_per_day       0
social_media_mins           0
study_mins                  0
physical_activity_days      0
sleep_hours                 0
sleep_quality               0
anxiety_score               0
depression_score            0
stress_level                0
happiness_score             0
focus_score                 0
high_risk_flag              0
device_type                 0
productivity_score          0
digital_dependence_score    0
dtype: int64

Duplicate IDs:
0

Dataset Shape:
(100000, 24)


##4. Standardize Categorical Columns

In [ ]:
categorical_columns = [
    "gender",
    "region",
    "income_level",
    "education_level",
    "daily_role",
    "device_type",
]

for col in categorical_columns:
    df[col] = (
        df[col]
        .astype(str)
        .str.strip()
        .str.title()
    )

print("Categorical values standardized.")

Categorical values standardized.


##5. Business Rule Validation

In [ ]:
rows_before = len(df)

df = df[
    (df["age"] >= 10) & (df["age"] <= 100)
]

df = df[
    (df["sleep_hours"] >= 0) & (df["sleep_hours"] <= 24)
]

df = df[
    (df["physical_activity_days"] >= 0)
    & (df["physical_activity_days"] <= 7)
]

rows_after  = len(df)
print(f"Business rules applied. Rows removed: {rows_before - rows_after}")

Business rules applied. Rows removed: 0


##6. Create Analytical Features

In [ ]:
# Age Group
df["age_group"] = pd.cut(
    df["age"],
    bins=[0, 18, 25, 35, 50, 100],
    labels=["Teen", "Young Adult", "Adult", "Mid Age", "Senior"],
)

# Screen Time Category
df["screen_time_category"] = pd.cut(
    df["device_hours_per_day"],
    bins=[0, 3, 6, 9, 24],
    labels=["Low", "Moderate", "High", "Very High"],
)

# Sleep Category
df["sleep_category"] = pd.cut(
    df["sleep_hours"],
    bins=[0, 5, 7, 9, 24],
    labels=["Poor Sleep", "Average Sleep", "Healthy Sleep", "Oversleep"],
)

# Mental Health Risk Category
# [FIX] Upper bin extended from 10 to 30 to cover the actual data
# range of anxiety_score (observed max = 27.3). The original cap of
# 10 silently produced NaN for ~20 % of rows because pd.cut excludes
# values above the highest bin edge.
df["mental_health_risk"] = pd.cut(
    df["anxiety_score"],
    bins=[0, 3, 6, 30],          # 30 safely covers PHQ-7 / GAD-7 max
    labels=["Low", "Medium", "High"],
    include_lowest=True,          # include anxiety_score == 0 in the Low bin
)

# Productivity Category
df["productivity_category"] = pd.cut(
    df["productivity_score"],
    bins=[0, 40, 60, 80, 100],
    labels=["Low", "Average", "Good", "Excellent"],
)

##7. Create Digital Wellness Score

In [ ]:
# [FIX] The three source columns are on different scales:
#   happiness_score : 0 – 10
#   focus_score     : 0 – 100
#   sleep_quality   : 0 – 10
#
# Without normalization focus_score (×10 magnitude) would dominate
# the average and make happiness/sleep effectively irrelevant.
# Each column is divided by its maximum scale value so all three
# contribute equally, and the result is rescaled to 0–10 for
# readability in dashboards.

df["digital_wellness_score"] = (
    (df["happiness_score"] / 10)
    + (df["focus_score"]   / 100)
    + (df["sleep_quality"] / 10)
) / 3 * 10     # rescale composite back to a 0–10 range

df["digital_wellness_score"] = df["digital_wellness_score"].round(2)


##8. Round All Float Columns to 2 Decimal Places

In [ ]:
# Applied after all feature engineering so the rounding covers
# both original float columns and newly created ones.

float_cols = df.select_dtypes(include="float64").columns.tolist()
df[float_cols] = df[float_cols].round(2)

print(f"Rounded {len(float_cols)} float columns to 2 decimal places:")
print(float_cols)

Rounded 10 float columns to 2 decimal places:
['device_hours_per_day', 'sleep_hours', 'sleep_quality', 'anxiety_score', 'stress_level', 'happiness_score', 'focus_score', 'productivity_score', 'digital_dependence_score', 'digital_wellness_score']


##9. Silver Layer Audit Summary

In [ ]:
print("===== SILVER LAYER SUMMARY =====")
print(f"Rows    : {df.shape[0]}")
print(f"Columns : {df.shape[1]}")

print("\nAge Groups:")
print(df["age_group"].value_counts())

print("\nScreen Time Categories:")
print(df["screen_time_category"].value_counts())

print("\nMental Health Risk — NaN check (should be 0 after fix):")
print(df["mental_health_risk"].isna().sum())

print("\nDigital Wellness Score — sample stats:")
print(df["digital_wellness_score"].describe().round(2))
print("================================")


===== SILVER LAYER SUMMARY =====
Rows    : 100000
Columns : 30

Age Groups:
age_group
Adult          30599
Young Adult    27207
Mid Age        18671
Teen           13920
Senior          9603
Name: count, dtype: int64

Screen Time Categories:
screen_time_category
High         35402
Moderate     33729
Very High    25972
Low           4897
Name: count, dtype: int64

Mental Health Risk — NaN check (should be 0 after fix):
0

Digital Wellness Score — sample stats:
count    100000.00
mean          4.42
std           1.45
min           0.33
25%           3.43
50%           4.56
75%           5.54
max           8.33
Name: digital_wellness_score, dtype: float64


##10. Save Silver Dataset

In [ ]:
df.to_csv(SILVER_FILE_PATH, index=False)
print(f"Silver Layer file saved to: {SILVER_FILE_PATH}")

Silver Layer file saved to: Silver_Layer/Digital_Lifestyle_Benchmark_Silver.csv


#Summary & Next Steps

The Silver Layer is complete:



*   Data validated
*   Business rules enforced
*   Categorical values standardized
*   Analytical features created
*   Dataset prepared for reporting and visualization


Next stage:

Gold Layer

The Gold Layer will contain KPI tables, aggregated datasets, dashboard-ready views, and Power BI reporting models.